# Muallimi soniy lessons: transcripts on Colab

Transcribes the 73 narrated *Muallimi soniy* lessons (~92 min) with word timestamps for line-by-line
text tracking (`docs/TAJWEED_PLAN.md` P2). About 10–15 minutes on a T4 GPU.

**Run:** Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**. Results go to Google Drive
(`MyDrive/quranayat_muallimi`), so a disconnect loses nothing: run all again and finished lessons are skipped.
The last cell downloads one zip (transcripts + pause profiles, no audio). Send it back to Claude.

## 1. GPU check

In [ ]:
import subprocess

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
)
assert gpu.returncode == 0, "No GPU: Runtime -> Change runtime type -> T4 GPU, then run again."
print(gpu.stdout.strip())

## 2. Google Drive for results

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/quranayat_muallimi"  # transcripts and pause profiles: no audio
os.makedirs(OUT, exist_ok=True)
os.environ["TAJWEED_CACHE"] = OUT
os.makedirs(f"{OUT}/transcripts", exist_ok=True)
done = [f for f in os.listdir(f"{OUT}/transcripts") if f.endswith(".json")]
print(f"{len(done)} of 73 lessons already transcribed")

## 3. Install and get the code

In [ ]:
!pip -q install "faster-whisper>=1.1.1,<2" nvidia-cublas-cu12 "nvidia-cudnn-cu12>=9,<10"
!rm -rf /content/repo && git clone -q --depth 1 --branch feat/tajweed https://github.com/otabek-rizakulov-cntxt/QuranAyat_bot.git /content/repo

Colab's GPU image ships CUDA libraries newer than the CUDA 12 build faster-whisper uses
(`Library libcublas.so.12 is not found`). This preloads the CUDA 12 cuBLAS and cuDNN 9 libraries.

In [ ]:
import ctypes
import glob

import nvidia

libs = [
    p
    for d in nvidia.__path__
    for sub in ("cublas", "cudnn")
    for p in sorted(glob.glob(f"{d}/{sub}/lib/lib*.so*"))
]


def load(path):
    try:
        ctypes.CDLL(path, mode=ctypes.RTLD_GLOBAL)
        return True
    except OSError:
        return False


for _ in range(3):  # some libraries need others loaded first
    ok = [load(p) for p in libs]
print(sum(ok), "of", len(libs), "CUDA 12 libraries loaded")

## 4. Transcribe

Downloads each lesson to Colab's temporary disk (never Drive) and transcribes it. In-process, so the CUDA
libraries above are used. Finished lessons are skipped.

In [ ]:
import sys

sys.path.insert(0, "/content/repo")
os.chdir("/content/repo")
from scripts.tajweed import transcribe_muallimi as tm  # noqa: E402

tm.MP3_DIR = "/content/mp3"  # audio stays off Drive
sys.argv = ["transcribe_muallimi"]
tm.main()

## 5. Download the results

In [ ]:
import shutil

from google.colab import files

stage = "/content/quranayat_muallimi"
shutil.rmtree(stage, ignore_errors=True)
shutil.copytree(OUT, stage)
zip_path = shutil.make_archive(stage, "zip", stage)
n = len([f for f in os.listdir(f"{stage}/transcripts") if f.endswith(".json")])
print(zip_path, round(os.path.getsize(zip_path) / 1e6, 1), "MB,", n, "lessons")
files.download(zip_path)